# Workshop: Wie LLMs Wörter wählen – Ansätze & Nachhaltigkeit

In diesem Notebook werden Sie

* mit verschiedenen Decoding-Strategien der autoregressiven Textgenerierung experimentieren: Greedy Search, Beam Search, Sampling (mit Temperature), Top-k Sampling, Top-p Sampling.
* mit `CodeCarbon` arbeiten, um CO2-Emissionen bei der Textgenerierung zu tracken.

In [33]:
### Hier bitte nichts ändern ###
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import pandas as pd
import accelerate

In [34]:
### Hier bitte nichts ändern ###
!pip install codecarbon
from codecarbon import EmissionsTracker

Auf den folgenden Seiten können Informationen zur Motivation, Methodologie und Anwendung von `CodeCarbon` nachgelesen werden:

* Motivation: https://docs.codecarbon.io/latest/explanation/why/
* Methodologie: https://docs.codecarbon.io/latest/explanation/methodology/
* Anwendung: https://docs.codecarbon.io/latest/tutorials/first-tracking/


Wir arbeiten mit einem "kleinen" Sprachmodell, das frei verfügbar ist.
Informationen zum Sprachmodell und einen Quickstart mit Codebeispiel findest Du hier:
http://huggingface.co/Qwen/Qwen2.5-3B-Instruct

In [35]:
### Hier bitte nichts ändern ###
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Im Folgenden haben Sie die Möglichkeit eine beliebige Prompt zu definieren. Diese wird anschließend in eine Form gebracht, die als Modelinput geeignet ist.

In [36]:
prompt = "Was ist 0,34+0,76?" # Zwischen den Anführungsstrichen Prompt definieren
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


Für die Textgenerierung und die Bestimmung der Emissionen bei der Textgenerierung nutzen wir die folgende Funktion.

In [37]:
### Hier bitte nichts ändern ###
def run_with_tracking(name, max_new_tokens, **gen_kwargs):
    tracker = EmissionsTracker(
        project_name=f"hf-generate-{name}",
        output_file="emissions.csv",
        measure_power_secs=1,
        log_level="error",
    )
    tracker.start()

    t0 = time.time()
    out = model.generate(**model_inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
    dt = time.time() - t0

    emissions_kg = tracker.stop()  # kg CO2eq
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    print("\n" + "="*18, name, "="*18)
    print(text)
    print(f"\nZeit: {dt:.2f}s | Emissionen: {emissions_kg*1000:.2f} gCO2e")

Wir generieren Text zunächst nicht-deterministisch per Sampling.

Variieren Sie die Zahl der maximal generierten Tokens `max_new_tokens` ab. (Um die Laufzeit gering zu halten, wählen Sie Werte <60.)

Was stellen Sie fest?

In [53]:
# 1) Sampling (Temperature)
run_with_tracking("Sampling", max_new_tokens=20, do_sample=True, temperature=0.9) # Sie können in dieser Zeile den Wert von max_new_tokens abändern


================== Sampling ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Addition von 0,34 und 0,76 ergibt 1,1

Zeit: 36.46s | Emissionen: 0.12 gCO2e


In [54]:
# 1) Sampling (Temperature)
run_with_tracking("Sampling", max_new_tokens=40, do_sample=True, temperature=0.9) # Sie können in dieser Zeile den Wert von max_new_tokens abändern


================== Sampling ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Um 0,34 und 0,76 zu addieren, addierst du die Zehnerstellen und die Einmaleinungen einfach:

0,34 + 0

Zeit: 57.43s | Emissionen: 0.18 gCO2e


Wir generieren Text nun deterministisch per Greedy Search und per Beam Search.

Variieren Sie bei Beam Search die Zahl der berücksichtigten Beams `num_beams`. (Um die Laufzeit gering zu halten, wählen Sie Werte <5.)

Welche Unterschiede stellen Sie hinsichtlich Laufzeit und Emissionen fest?

In [40]:
# 2) Greedy
run_with_tracking("Greedy", max_new_tokens=20, do_sample=False)


================== Greedy ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Addition von 0,34 und 0,76 ergibt 1,1

Zeit: 43.29s | Emissionen: 0.14 gCO2e


In [41]:
# 3) Beam Search
run_with_tracking("Beam", max_new_tokens=20, do_sample=False, num_beams=2, early_stopping=True)


================== Beam ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Addition von 0,34 und 0,76 ergibt 1,1

Zeit: 81.74s | Emissionen: 0.26 gCO2e


In [42]:
# 3) Beam Search
run_with_tracking("Beam", max_new_tokens=20, do_sample=False, num_beams=3, early_stopping=True)


================== Beam ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Addition von 0,34 und 0,76 ergibt 1,1

Zeit: 100.32s | Emissionen: 0.32 gCO2e


Experimentieren Sie nun auch mit den restlichen Decoding-Strategien und vergleichen Sie die Ergebnisse.


In [43]:
# 4) Top-k Sampling
run_with_tracking("Top-k (k=10)", max_new_tokens=20, do_sample=True, top_k=10)


================== Top-k (k=10) ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Addition von 0,34 und 0,76 ergibt 1,1

Zeit: 36.43s | Emissionen: 0.12 gCO2e


In [44]:
# 4) Top-k Sampling
run_with_tracking("Top-k (k=50)", max_new_tokens=20, do_sample=True, top_k=50)


================== Top-k (k=50) ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
0,34 + 0,76 = 1,10

Das Ergebnis

Zeit: 36.32s | Emissionen: 0.12 gCO2e


In [45]:
# 5) Top-p Sampling
run_with_tracking("Top-p (p=0.9)", max_new_tokens=20, do_sample=True, top_p=0.9)


================== Top-p (p=0.9) ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Summe von 0,34 und 0,76 ist 1,1

Zeit: 40.07s | Emissionen: 0.13 gCO2e


In [46]:
# 5) Top-p Sampling
run_with_tracking("Top-p (p=0.95)", max_new_tokens=20, do_sample=True, top_p=0.95)


================== Top-p (p=0.95) ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
0,34 + 0,76 = 1,10

Dies können wir

Zeit: 36.46s | Emissionen: 0.12 gCO2e


Während der Textgenerierung wurden von `CodeCarbon` `csv`-Dateien angelegt. Im Folgenden lesen wir eine der Dateien per `pd.read_csv` ein und untersuchen die Inhalte.

Welche Informationen liest `CodeCarbon` zur Berechnung der Emissionen aus?

In [47]:
### Hier bitte nichts ändern ###
row = pd.read_csv("emissions.csv").iloc[-1]

print("CPU:", row["cpu_model"])
print("CPU cores:", row["cpu_count"])
print("Country:", row.get("country_name"))
print("Region:", row.get("region"))

CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
CPU cores: 2
Country: United States
Region: district of columbia


`CodeCarbon` kann auch im "Offline"-Modus verwendet werden, zum Beispiel wenn kein Internet verfügbar ist und daher der Standort nicht abgerufen werden kann. In diesem Fall ist eine manuelle Spezifikation des Standorts via ISO Code nötig.

In [48]:
### Hier bitte nichts ändern ###

from codecarbon import OfflineEmissionsTracker

def run_with_tracking_ISO(name, max_new_tokens, iso, **gen_kwargs):
    tracker = OfflineEmissionsTracker(
        project_name=f"hf-generate-{name}", # Dokumentation des Emission Trackings
        output_file="emissions.csv",
        measure_power_secs=1, # Messintervall
        log_level="error",
        country_iso_code=iso
    )
    tracker.start()

    t0 = time.time()
    out = model.generate(**model_inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
    dt = time.time() - t0

    emissions_kg = tracker.stop()  # kg CO2eq
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    print("\n" + "="*18, name, "="*18)
    print(text)
    print(f"\nZeit: {dt:.2f}s | Emissionen: {emissions_kg*1000:.2f} gCO2e")

Testen Sie verschiedene Länder-ISO-Codes `iso` (https://de.wikipedia.org/wiki/ISO-3166-1-Kodierliste). Was stellen Sie fest?

In [49]:
run_with_tracking_ISO("Sampling", max_new_tokens=20, iso="DEU", do_sample=True, temperature=0.9) # Sie können in dieser Zeile den Wert von iso abändern


================== Sampling ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Addition von 0,34 und 0,76 ergibt 1,1

Zeit: 36.86s | Emissionen: 0.20 gCO2e


In [50]:
run_with_tracking_ISO("Sampling", max_new_tokens=20, iso="FRA", do_sample=True, temperature=0.9) # Sie können in dieser Zeile den Wert von iso abändern


================== Sampling ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Um die beiden Zahlen 0,34 und 0,76 zu addieren,

Zeit: 36.78s | Emissionen: 0.03 gCO2e


In [51]:
run_with_tracking_ISO("Sampling", max_new_tokens=20, iso="CHN", do_sample=True, temperature=0.9) # Sie können in dieser Zeile den Wert von iso abändern


================== Sampling ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
Die Addition von 0,34 und 0,76 ergibt 1,1

Zeit: 36.96s | Emissionen: 0.31 gCO2e


In [52]:
run_with_tracking_ISO("Sampling", max_new_tokens=20, iso="USA", do_sample=True, temperature=0.9) # Sie können in dieser Zeile den Wert von iso abändern


================== Sampling ==================
system
You are a helpful assistant.
user
Was ist 0,34+0,76?
assistant
0,34 + 0,76 = 1,10

Das Ergebnis

Zeit: 36.46s | Emissionen: 0.20 gCO2e
